<a href="https://colab.research.google.com/github/Chandrani45/AI-COURSE-RECOMMENDER-PROJECT-WORK-/blob/main/Copy_of_Chandrani_Sengupta_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LOADING THE LIBRARIES

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.0 MB/s eta 0:00:00


MODULE 0
Loading Groq

In [3]:
import os
from groq import Groq

In [4]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [5]:
client = Groq()

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Explain why Groq's inference is fast in 3 sentences."},
    ],
)

print(response.choices[0].message.content)

Groq's inference is fast due to its unique tensor streaming processor architecture, which allows for efficient processing of large amounts of data. This architecture enables the system to handle complex machine learning models and process them quickly, resulting in high inference speeds. By optimizing data transfer and processing, Groq's technology reduces latency and increases throughput, making it an attractive solution for applications requiring fast and accurate inference.


LOADING THE COURSE DATASET

In [7]:
course=pd.read_csv('/content/coursera_course_dataset_v3.csv')

PRINTING THE FIRST 5 ROWS

In [8]:
course.head()

,Unnamed: 0,Title,Organization,Skills,Ratings,course_url,course_students_enrolled,course_description,Review Count,Difficulty,Type,Duration
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,https://www.coursera.org/professional-certific...,"700,909",Google Cloud Fundamentals: Core Infrastructure...,20K,Beginner,Professional Certificate,3 - 6 Months
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,https://www.coursera.org/professional-certific...,"229,865",Prepare for a new career in the high-growth fi...,137K,Beginner,Professional Certificate,3 - 6 Months
2,3,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,https://www.coursera.org/professional-certific...,"29,702",Prepare-se para uma nova carreira no campo de ...,100K,Beginner,Professional Certificate,3 - 6 Months
3,4,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,https://www.coursera.org/professional-certific...,"239,622",Prepare for a career in the high-growth field ...,120K,Beginner,Professional Certificate,3 - 6 Months
4,5,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,https://www.coursera.org/professional-certific...,"384,238",This course is the eighth course in the Google...,23K,Beginner,Professional Certificate,3 - 6 Months


We drop the unnamed column as it is of no use

In [9]:
course.rename(columns={'Unnamed: 0':'Course_id'},inplace=True)

In [10]:
course.head()

,Course_id,Title,Organization,Skills,Ratings,course_url,course_students_enrolled,course_description,Review Count,Difficulty,Type,Duration
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,https://www.coursera.org/professional-certific...,"700,909",Google Cloud Fundamentals: Core Infrastructure...,20K,Beginner,Professional Certificate,3 - 6 Months
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,https://www.coursera.org/professional-certific...,"229,865",Prepare for a new career in the high-growth fi...,137K,Beginner,Professional Certificate,3 - 6 Months
2,3,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,https://www.coursera.org/professional-certific...,"29,702",Prepare-se para uma nova carreira no campo de ...,100K,Beginner,Professional Certificate,3 - 6 Months
3,4,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,https://www.coursera.org/professional-certific...,"239,622",Prepare for a career in the high-growth field ...,120K,Beginner,Professional Certificate,3 - 6 Months
4,5,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,https://www.coursera.org/professional-certific...,"384,238",This course is the eighth course in the Google...,23K,Beginner,Professional Certificate,3 - 6 Months


In [11]:
course.columns

Index(['Course_id', 'Title', 'Organization', 'Skills', 'Ratings', 'course_url',
       'course_students_enrolled', 'course_description', 'Review Count',
       'Difficulty', 'Type', 'Duration'],
      dtype='object')

In [12]:
course.columns=course.columns.str.lower()


In [13]:
course.head()

,course_id,title,organization,skills,ratings,course_url,course_students_enrolled,course_description,review count,difficulty,type,duration
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,https://www.coursera.org/professional-certific...,"700,909",Google Cloud Fundamentals: Core Infrastructure...,20K,Beginner,Professional Certificate,3 - 6 Months
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,https://www.coursera.org/professional-certific...,"229,865",Prepare for a new career in the high-growth fi...,137K,Beginner,Professional Certificate,3 - 6 Months
2,3,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,https://www.coursera.org/professional-certific...,"29,702",Prepare-se para uma nova carreira no campo de ...,100K,Beginner,Professional Certificate,3 - 6 Months
3,4,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,https://www.coursera.org/professional-certific...,"239,622",Prepare for a career in the high-growth field ...,120K,Beginner,Professional Certificate,3 - 6 Months
4,5,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,https://www.coursera.org/professional-certific...,"384,238",This course is the eighth course in the Google...,23K,Beginner,Professional Certificate,3 - 6 Months


In [14]:
#print rows and columns
course.shape

(623, 12)

The data has 623 rows and 11 columns

In [15]:
role_skills={'Data Scientist': ['python', 'statistics', 'machine learning', 'sql',
                           'data visualization', 'deep learning'], 'Data Analyst': ['excel', 'sql', 'statistics', 'data visualization', 'python'], 'ML Engineer':['python', 'machine learning', 'deep learning', 'sql', 'cloud'],'Data Engineer': ['python','sql', 'data wrangling', 'cloud', 'big data', 'apis'], 'Business Analyst': ['excel', 'sql', 'statistics', 'data visualization',
                           'communication'],'BI Developer': ['sql', 'data visualization', 'excel', 'statistics', 'power bi'],'AI Researcher':['python', 'deep learning', 'machine learning',
                           'mathematics', 'nlp'], 'Backend Developer': ['python', 'sql', 'apis', 'cloud', 'git'],'MLOps Engineer':['python', 'machine learning', 'cloud', 'docker', 'mlops'], 'NLP Engineer':['python', 'machine learning', 'deep learning', 'nlp',
                           'statistics']



}

In [16]:
role_skills['Data Scientist']

['python',
 'statistics',
 'machine learning',
 'sql',
 'data visualization',
 'deep learning']

In [17]:
course.isnull().sum()

,0
course_id,0
title,0
organization,0
skills,0
ratings,0
course_url,220
course_students_enrolled,236
course_description,221
review count,0
difficulty,0


In [18]:
course.duplicated().sum()

np.int64(0)

No duplicates as such

In [19]:
course.drop(columns=['organization','ratings','course_url' ,	'course_students_enrolled' ,'course_description', 'review count','difficulty','type','duration'],axis=1,inplace=True)

In [20]:
course.isnull().sum()

,0
course_id,0
title,0
skills,0


In [21]:
course.head()

,course_id,title,skills
0,0,Google Cybersecurity,"Network Security, Python Programming, Linux, ..."
1,1,Google Data Analytics,"Data Analysis, R Programming, SQL, Business C..."
2,3,Google Project Management:,"Project Management, Strategy and Operations, ..."
3,4,IBM Data Science,"Python Programming, Data Science, Machine Lea..."
4,5,Google Digital Marketing & E-commerce,"Digital Marketing, Marketing, Marketing Manag..."


In [ ]:
course.shape

In [22]:
#cleaning the skills dataset
import re
def clean(text):
  return re.sub(r'[^a-z0-9]','',text.str.lower())
  return re.sub(r'\s+',"",text).strip()

In [23]:
import re

def clean(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s).lower())
    return re.sub(r"\s+", " ", s).strip()

# build the search text from Title + Skills  (note the " " between them)
course['text'] = (course['title'] + " " + course['skills']).apply(clean)

# keep a clean LIST of skills — M6 needs this
course['skills'] = course['skills'].fillna("").apply(
    lambda s: [x.strip().lower() for x in str(s).split(",") if x.strip()])

course[['course_id', 'title', 'skills', 'text']].head(3)

,course_id,title,skills,text
0,0,Google Cybersecurity,"[network security, python programming, linux, ...",google cybersecurity network security python p...
1,1,Google Data Analytics,"[data analysis, r programming, sql, business c...",google data analytics data analysis r programm...
2,3,Google Project Management:,"[project management, strategy and operations, ...",google project management project management s...


In [24]:
course.head()

,course_id,title,skills,text
0,0,Google Cybersecurity,"[network security, python programming, linux, ...",google cybersecurity network security python p...
1,1,Google Data Analytics,"[data analysis, r programming, sql, business c...",google data analytics data analysis r programm...
2,3,Google Project Management:,"[project management, strategy and operations, ...",google project management project management s...
3,4,IBM Data Science,"[python programming, data science, machine lea...",ibm data science python programming data scien...
4,5,Google Digital Marketing & E-commerce,"[digital marketing, marketing, marketing manag...",google digital marketing e commerce digital ma...


In [25]:
course['text'].iloc[0]

'google cybersecurity network security python programming linux cloud computing algorithms audit computer programming computer security incident management cryptography databases leadership and management network architecture risk management sql'

In [30]:
target_role = input('Enter a role: ').strip()
assert target_role in role_skills, f"'{target_role}' is not in the role list. Pick one of: {list(role_skills)}"

Enter a role: NLP Engineer


In [31]:
target_role

'NLP Engineer'

In [32]:
current_skills = [s.strip().lower() for s in input('Enter your current skills (comma-separated): ').split(",") if s.strip()]
have = set(current_skills)


Enter your current skills (comma-separated): machine learning


In [33]:
have

{'machine learning'}

In [34]:
skill_gap = [s for s in role_skills[target_role] if s.lower() not in have]
query_text = " ".join(skill_gap)
print("skill_gap:", skill_gap)

skill_gap: ['python', 'deep learning', 'nlp', 'statistics']


In [35]:
query_text

'python deep learning nlp statistics'

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [37]:
tfidf=TfidfVectorizer(stop_words='english')

In [38]:
course_vector=tfidf.fit_transform(course['text']).toarray()

In [39]:
from sklearn.metrics.pairwise import cosine_similarity

In [42]:
def recommend(query_text):
  query_vector=tfidf.transform([query_text])
  similarity=cosine_similarity(course_vector,query_vector).flatten()
  course_list=course.copy()
  course_list['score']=similarity
  course_list=course_list.sort_values(by='score',ascending=False)
  return course_list.head(10)

In [43]:
result_tfdif=recommend(query_text)

In [44]:
result_tfdif

,course_id,title,skills,text,score
409,415,NLP: Twitter Sentiment Analysis,"[machine learning, natural language processing...",nlp twitter sentiment analysis machine learnin...,0.442652
464,470,AI for Medicine,"[machine learning, machine learning algorithms...",ai for medicine machine learning machine learn...,0.338454
288,294,Deep Neural Networks with PyTorch,"[human learning, machine learning, deep learni...",deep neural networks with pytorch human learni...,0.326782
97,101,Neural Networks and Deep Learning,"[artificial neural networks, deep learning, ma...",neural networks and deep learning artificial n...,0.326431
93,97,IBM Machine Learning,"[machine learning, machine learning algorithms...",ibm machine learning machine learning machine ...,0.312055
87,91,Advanced Learning Algorithms,"[applied machine learning, machine learning, m...",advanced learning algorithms applied machine l...,0.286911
7,9,Machine Learning,"[machine learning, machine learning algorithms...",machine learning machine learning machine lear...,0.283441
17,21,Deep Learning,"[machine learning, deep learning, artificial n...",deep learning machine learning deep learning a...,0.278952
172,178,Improving Deep Neural Networks: Hyperparameter...,"[applied machine learning, deep learning, mach...",improving deep neural networks hyperparameter ...,0.278756
66,70,Natural Language Processing,"[natural language processing, machine learning...",natural language processing natural language p...,0.278586


In [45]:
from sentence_transformers import SentenceTransformer,util

In [46]:
model=SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [47]:
course_embedding=model.encode(course['text'].tolist(),convert_to_tensor=False)

In [ ]:
#course_embedding

In [48]:
def recommend2(query_text):
  query_vector2=model.encode([query_text],convert_to_tensor=False)
  similarity=cosine_similarity(course_embedding,query_vector2).flatten()
  course_list=course.copy()
  course_list['score_embedding']=similarity
  course_list=course_list.sort_values(by='score_embedding',ascending=False)
  return course_list.head(10)

In [49]:
result_embedding=recommend2(query_text)

In [50]:
result_embedding

,course_id,title,skills,text,score_embedding
295,301,Natural Language Processing with Attention Models,"[artificial neural networks, deep learning, na...",natural language processing with attention mod...,0.621569
409,415,NLP: Twitter Sentiment Analysis,"[machine learning, natural language processing...",nlp twitter sentiment analysis machine learnin...,0.559481
368,374,Introduction to Machine Learning,"[artificial neural networks, machine learning,...",introduction to machine learning artificial ne...,0.535411
66,70,Natural Language Processing,"[natural language processing, machine learning...",natural language processing natural language p...,0.531863
27,31,Generative AI with Large Language Models,"[machine learning, natural language processing...",generative ai with large language models machi...,0.508301
464,470,AI for Medicine,"[machine learning, machine learning algorithms...",ai for medicine machine learning machine learn...,0.457548
175,181,AI for Good,"[artificial neural networks, computer vision, ...",ai for good artificial neural networks compute...,0.456137
67,71,DeepLearning.AI TensorFlow Developer,"[machine learning, tensorflow, deep learning, ...",deeplearning ai tensorflow developer machine l...,0.446086
35,39,Generative AI Fundamentals,"[machine learning, natural language processing]",generative ai fundamentals machine learning na...,0.436551
288,294,Deep Neural Networks with PyTorch,"[human learning, machine learning, deep learni...",deep neural networks with pytorch human learni...,0.420564


In [51]:
def is_relevant(course_skills,skill_gap):
  course_skills=set(course_skills)
  skill_gap=set(skill_gap)
  return len(course_skills & skill_gap)>0

In [52]:
def precision_at_k(recommendations,skill_gap,k):
  relevant=0
  for _,row in recommendations.head(k).iterrows():
    if is_relevant(row['skills'],skill_gap):
      relevant+=1

  return relevant/k

In [53]:
skill_gap

['python', 'deep learning', 'nlp', 'statistics']

In [55]:
precisiontfdif=precision_at_k(result_tfdif,skill_gap,10)
print(precisiontfdif)

0.9


In [56]:
precision_embedding=precision_at_k(result_embedding,skill_gap,10)
print(precision_embedding)

0.5


In [63]:
def evaluate_role(target_role, current_skills, k=10):
    have = {s.strip().lower() for s in current_skills}
    skill_gap = [s for s in role_skills[target_role] if s.lower() not in have]
    query_text = " ".join(skill_gap)

    tfidf_recs = recommend(query_text)
    emb_recs   = recommend2(query_text)

    p_tfidf = precision_at_k(tfidf_recs, skill_gap, k)
    p_emb   = precision_at_k(emb_recs,  skill_gap, k)

    print(f"\n=== {target_role} | have={current_skills} ===")
    print("skill gap:", skill_gap)
    print("TF-IDF top-5:", tfidf_recs['title'].head(5).tolist())
    print("Embed  top-5:", emb_recs['title'].head(5).tolist())
    print(f"Precision@{k} -> TF-IDF={p_tfidf:.2f}  Embedding={p_emb:.2f}")
    return {"role": target_role, "p_tfidf": p_tfidf, "p_emb": p_emb}

scenarios = [
    ("Data Scientist", ["python"]),
    ("Data Analyst",   ["excel"]),
    ("NLP Engineer",   ["python", "machine learning"]),
]

rows = [evaluate_role(r, s) for r, s in scenarios]
import pandas as pd
pd.DataFrame(rows)



=== Data Scientist | have=['python'] ===
skill gap: ['statistics', 'machine learning', 'sql', 'data visualization', 'deep learning']
TF-IDF top-5: ['Machine Learning on Google Cloud', 'Machine Learning', 'IBM Machine Learning', 'Applying Machine Learning to your Data with Google Cloud', 'Deep Neural Networks with PyTorch']
Embed  top-5: ['Data Analytics for Lean Six Sigma', 'From Data to Insights with Google Cloud', 'Share Data Through the Art of Visualization', '1. 基礎知識：データはあらゆるところにある', 'Understanding and Visualizing Data with Python']
Precision@10 -> TF-IDF=1.00  Embedding=1.00

=== Data Analyst | have=['excel'] ===
skill gap: ['sql', 'statistics', 'data visualization', 'python']
TF-IDF top-5: ['Data Visualization with Python', 'SQL für Data Science', 'Data Analysis and Visualization Foundations', 'Statistics for Data Science with Python', 'Understanding and Visualizing Data with Python']
Embed  top-5: ['Understanding and Visualizing Data with Python', 'Data Visualization with Pytho

,role,p_tfidf,p_emb
0,Data Scientist,1.0,1.0
1,Data Analyst,1.0,1.0
2,NLP Engineer,0.9,0.4


In [64]:
rows

[{'role': 'Data Scientist', 'p_tfidf': 1.0, 'p_emb': 1.0},
 {'role': 'Data Analyst', 'p_tfidf': 1.0, 'p_emb': 1.0},
 {'role': 'NLP Engineer', 'p_tfidf': 0.9, 'p_emb': 0.4}]

In [66]:
prompt=f"""The user wants to become a {target_role}

current skills:
{current_skills}

missing skills:
{
    skill_gap
}

recommended courses:

{
    result_tfdif['title'].head(10).tolist()
}

Create a learning roadmap in order
"""

In [67]:
response=client.chat.completions.create(model="llama-3.3-70b-versatile" , messages=[{'role':'user','content':prompt}])
print(response.choices[0].message.content)

To become an NLP Engineer, you need to acquire the missing skills: Python, Deep Learning, NLP, and Statistics. Here's a suggested learning roadmap in order:

**Phase 1: Fundamentals (Python and Statistics)**

1. **Python**: Start by learning Python basics. Although there's no specific Python course in the recommendations, you can find online resources like Codecademy, Python.org, or DataCamp.
2. **Statistics**: Since there's no specific statistics course in the recommendations, you can find online resources like Khan Academy, Coursera, or edX.

**Phase 2: Deep Learning**

1. **Neural Networks and Deep Learning**: Take this course to learn the fundamentals of deep learning.
2. **Deep Neural Networks with PyTorch**: This course will help you learn PyTorch, a popular deep learning framework, and apply your knowledge of neural networks.
3. **Deep Learning**: This course will further reinforce your understanding of deep learning concepts.

**Phase 3: NLP**

1. **Natural Language Processing*

In [68]:
prompt2=f"""The user wants to become a {target_role}

current skills:
{current_skills}

missing skills:
{
    skill_gap
}

recommended courses:

{
    result_embedding['title'].head(10).tolist()
}

Create a learning roadmap in order
"""

In [69]:
response2=client.chat.completions.create(model="llama-3.3-70b-versatile" , messages=[{'role':'user','content':prompt2}])
print(response2.choices[0].message.content)

To become an NLP Engineer, here's a suggested learning roadmap based on your current skills and the missing skills you need to acquire:

**Phase 1: Foundational Skills (3-6 months)**

1. **Statistics**: Start by learning the basics of statistics, which will help you understand the underlying concepts of machine learning and NLP. Online resources like Khan Academy, Coursera, or edX can provide a good introduction to statistics.
2. **Python**: Learn the fundamentals of Python programming, which is a crucial skill for NLP. Take online courses or tutorials that cover the basics of Python, such as data types, functions, and data structures.
3. **Introduction to Machine Learning**: Although you have some experience with machine learning, it's essential to revisit the basics and reinforce your understanding of the subject. Take the 'Introduction to Machine Learning' course to fill any gaps in your knowledge.

**Phase 2: Deep Learning and NLP Fundamentals (6-9 months)**

1. **Deep Learning**: 